In [1]:
import pandas as pd
import numpy as np
from embeddings import embed_text
from retrieval import create_index, search
from rerank import rerank
from generator import generate_answer
from evaluation import evaluate_faithfulness, evaluate_relevance

# Load documents
with open("../data/documents.txt", "r") as f:
    documents = [line.strip() for line in f.readlines()]

# Create embeddings
doc_embeddings = embed_text(documents)

# Create FAISS index
index = create_index(doc_embeddings)

test_questions = [
    "What is supervised learning?",
    "What are applications of machine learning?",
    "Explain reinforcement learning."
]

results = []

for question in test_questions:
    print(f"\nQuestion: {question}")

    query_embedding = embed_text([question])
    distances, indices = search(index, query_embedding, k=5)

    retrieved_docs = [documents[i] for i in indices[0]]
    reranked_docs = rerank(question, retrieved_docs)

    context = "\n".join(reranked_docs[:3])

    answer = generate_answer(question, context)

    faith_score, faith_text = evaluate_faithfulness(question, context, answer)
    rel_score, rel_text = evaluate_relevance(question, answer)

    results.append({
        "Question": question,
        "Answer": answer,
        "Faithfulness Score": faith_score,
        "Relevance Score": rel_score
    })

df = pd.DataFrame(results)
df.to_csv("../evaluation_report.csv", index=False)

print("\nEvaluation Complete. Report saved.")


ModuleNotFoundError: No module named 'embeddings'